# 自定义 Autograd 与 gradcheck

## 学习目标

使用 `torch.autograd.Function` 实现自定义算子，理解保存中间值、反向输入梯度和数值梯度检查。

## 概念模型

自定义 Function 的 `forward` 负责输出，`backward` 返回每个可微输入的梯度。生产代码优先使用 PyTorch 内置算子；只有确有必要时才自定义。

In [ ]:
import torch
from torch.autograd import Function

class SquarePlusOne(Function):
    @staticmethod
    def forward(ctx, inputs):
        ctx.save_for_backward(inputs)
        return inputs.square() + 1
    @staticmethod
    def backward(ctx, grad_output):
        (inputs,) = ctx.saved_tensors
        return grad_output * 2 * inputs

square_plus_one = SquarePlusOne.apply
x = torch.tensor([2.0], requires_grad=True)
square_plus_one(x).sum().backward()
print('value and gradient:', square_plus_one(x.detach()), x.grad)
assert x.grad.item() == 4

### 实验 1：用 gradcheck 验证自定义 backward

**实验目的**：对 `SquarePlusOne` 的解析梯度 `2*x` 做中心有限差分检查。gradcheck 使用 double 精度、很小扰动和严格容差，适合验证自定义 Function 的局部梯度。

`backward` 必须把上游 `grad_output` 乘局部导数；只返回 `2*x` 会在复合图中错误。非光滑点、随机 forward、原地修改和低精度都会让 gradcheck 不可靠。


In [ ]:
candidate = torch.randn(3, dtype=torch.double, requires_grad=True)
passed = torch.autograd.gradcheck(square_plus_one, (candidate,), eps=1e-6, atol=1e-4)
print('gradcheck:', passed)
assert passed

## 官方教程补充

**对应官方源文件：** `beginner_source/examples_autograd/polynomial_custom_function.py`、`intermediate_source/custom_function_double_backward_tutorial.rst`、`intermediate_source/custom_function_conv_bn_tutorial.py`

官方自定义 `autograd.Function` 把 forward 与 backward 视为严格的数学契约：`ctx.save_for_backward` 只保存反向所需张量，backward 接收上游梯度并为每个输入返回梯度或 None。用 float64 小输入运行 `gradcheck`；若需要二阶梯度，还要让 backward 本身可被 Autograd 记录并运行 `gradgradcheck`。原地修改必须特别声明并谨慎处理。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

说明 `ctx.save_for_backward` 保存什么、`backward` 为什么接收 `grad_output`，以及为什么 gradcheck 通常使用 double。

## 试一试

故意把 backward 中的 `2` 改成 `3`，观察 gradcheck 如何失败；再实现 `abs` 的自定义 backward 并讨论零点不可导。

## 常见错误与调试

返回梯度数量不匹配、忘记处理 broadcast、在 backward 中构建不必要的计算图、使用 float32 导致数值检查不稳定。